# Uji penerimaan SLR_09 v0.8.3 — menunggu review

50 acuan FINAL083 dan 685 label token masih usulan PENDING. Validasi review serta salin snapshot sumber terikat sebelum pemuatan NN. Uji ini belum dijalankan. Persetujuan putaran lama tidak mengesahkan acuan baru.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, shutil
ROOT = Path(os.environ['SLR09_PACKAGE_ROOT']).resolve()
WORK = Path(os.environ['SLR09_WORK_DIR']).resolve()
REVIEW = Path(os.environ['SLR09_REVIEW_XLSX']).resolve()
sys.path.insert(0, str(ROOT/'code/acceptance'))
from import_review_xlsx import convert
from evaluate_acceptance import validate, file_sha
bundle = convert(REVIEW, root=ROOT, validate_ready=True)
rows, preflight = validate(bundle, ROOT)
assert all(row['review']['status'] == 'APPROVED' for row in rows)
OUT = WORK/'acceptance_FINAL083'
OUT.mkdir(parents=True, exist_ok=False)
shutil.copy2(REVIEW, OUT/REVIEW.name)
for relative, expected in bundle['protocol']['binding']['files_sha256'].items():
    source = ROOT/relative
    assert file_sha(source) == expected
    target = OUT/'pre_inference_snapshot'/relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, target)
    assert file_sha(target) == expected
reviewed = OUT/'reviewed_bundle_frozen.json'
reviewed.write_text(json.dumps(bundle, ensure_ascii=False, indent=2)+'\n')
cmd = [sys.executable, str(ROOT/'code/acceptance/evaluate_acceptance.py'), '--root', str(ROOT),
    '--review', str(reviewed), '--sena-zip', os.environ['SLR09_SENA_ARCHIVE'],
    '--nara-zip', os.environ['SLR09_NARA_ARCHIVE'], '--output', str(OUT/'original_run'), '--environment-label', 'CODEX_CPU_REPRESENTATIVE']
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout: print(line, end='', flush=True)
proc.wait()
if proc.returncode: raise RuntimeError('Acceptance command failed; preserve original evidence.')
print('Inference completed. Read evaluation and review the actual quality decision before accepting the card.')
